# Ancestree on SQLite — quickstart & smoke test

This notebook exercises the rebuilt backend end to end (the `rebuild`
branch): **one SQLite database instead of node folders**, the redesigned
query vocabulary, node-level deduplication, Layer-2 chunk deltas, crash
forensics, prune/compact, read-only SQL, and the two output formats
(view-only HTML snapshot + grep-able sidecar export).

Everything below runs against a throwaway store in a temp directory.

In [1]:
import json
import random
import shutil
import tempfile
from pathlib import Path

from ancestree.store import LineageStore

workdir = Path(tempfile.mkdtemp(prefix="ancestree-notebook-"))
store = LineageStore(
    workdir / "demo",
    rules={"clean": ["ingest"], "model": ["clean"]},
    gen_triggers=["ingest"],
)
print("store root:", store.root.name)
print("policy: rules =", store.rules, "| dedup =", store.dedup, "| chunk =", store.chunk)
print("at rest:", sorted(p.name for p in store.root.iterdir()))

store root: demo
policy: rules = {'clean': ['ingest'], 'model': ['clean']} | dedup = True | chunk = True
at rest: ['ancestree.db', 'ancestree.db-shm', 'ancestree.db-wal']


## A three-step pipeline
The write ergonomics are unchanged: real paths via `/`, metadata via `add_meta`.

In [2]:
with store.create_node(step_type="ingest") as ingest:
    (ingest / "raw.csv").write_text("id,value\n1,10\n2,20\n3,30\n")
    ingest.add_meta("rows", 3)

with store.create_node(step_type="clean", parent=ingest) as clean:
    [raw] = store.from_parent(clean, "raw.csv")
    rows = raw.read_text().strip().splitlines()
    (clean / "clean.csv").write_text("\n".join(rows[:1] + rows[2:]))
    clean.add_meta("rows", 2)
    clean.add_meta("dropped", 1, group="Quality")

with store.create_node(step_type="model", parent=clean) as model:
    (model / "model.bin").write_bytes(random.Random(1).randbytes(50_000))
    model.add_meta("accuracy", 0.94, group="Metrics")
    model.add_meta("params", {"lr": 0.01, "depth": 4})

print("created:", ingest.node_id, "->", clean.node_id, "->", model.node_id)
print("nodes are rows — no folders:", sorted(p.name for p in store.root.iterdir()))

created: 42060f53 -> 9c3861d5 -> 6db6d0e1
nodes are rows — no folders: ['.scratch', 'ancestree.db', 'ancestree.db-shm', 'ancestree.db-wal']


## The query vocabulary
`find` / `latest` / `lineage` / `ancestors` / `children` — equality, lambdas and `parent_id` filters, all answered by SQL.

In [3]:
best = store.latest(step_type="model")
print("latest model:", best.node_id, "accuracy =", best.metadata["accuracy"]["value"])
print("lineage:", " -> ".join(n.step_type for n in store.lineage(best)))
print("good runs:", [n.node_id for n in store.find(accuracy=lambda a: a is not None and a > 0.9)])
print("ancestors with rows:", [n.step_type for n in store.ancestors(best, rows=lambda r: r is not None)])
print("children of ingest:", [n.step_type for n in store.children(ingest)])
print("roots:", [n.step_type for n in store.find(parent_id=[])])
print("provenance keys:", sorted(best.provenance))

latest model: 6db6d0e1 accuracy = 0.94
lineage: ingest -> clean -> model
good runs: ['6db6d0e1']
ancestors with rows: ['ingest', 'clean']
children of ingest: ['clean']
roots: ['ingest']
provenance keys: ['git_branch', 'git_commit', 'git_dirty', 'platform', 'python_version', 'user']


## Node-level deduplication
A content-identical rerun is never stored twice: the handle rebinds onto the existing node.

In [4]:
with store.create_node(step_type="model", parent=clean) as rerun:
    (rerun / "model.bin").write_bytes(random.Random(1).randbytes(50_000))
    rerun.add_meta("accuracy", 0.94, group="Metrics")
    rerun.add_meta("params", {"lr": 0.01, "depth": 4})

print("rerun adopted:", rerun.node_id == model.node_id)
print("model nodes in store:", len(store.find(step_type="model")))

rerun adopted: True
model nodes in store: 1


## Layer-2 dedup on near-duplicate artifacts
Versions differing by scattered in-place edits share almost nothing chunk-for-chunk — the resemblance/delta layer stores them as small deltas.

In [5]:
payload = bytearray(random.Random(2).randbytes(150_000))
for version in range(4):
    rng = random.Random(3 + version)
    for _ in range(1_500):
        payload[rng.randrange(len(payload))] = rng.randrange(256)
    with store.create_node(step_type="ingest") as node:
        (node / "state.bin").write_bytes(bytes(payload))
        node.add_meta("version", version)

stats = store.stats()
print(f"logical bytes: {stats['logical_bytes']:,}")
print(f"stored chunk bytes: {stats['chunk_stored_bytes']:,}")
print("dedup ratio:", stats["dedup_ratio"])
latest_version = store.latest(version=lambda v: v is not None)
assert (latest_version / "state.bin").read_bytes() == bytes(payload)
print("byte-exact read-back: OK")

logical bytes: 650,042
stored chunk bytes: 447,152
dedup ratio: 1.454


byte-exact read-back: OK


## Crash forensics
A block that raises keeps its partial output, flagged unhealthy and searchable — partial work is evidence.

In [6]:
try:
    with store.create_node(step_type="ingest") as doomed:
        (doomed / "partial.csv").write_text("half-written")
        raise RuntimeError("simulated mid-run failure")
except RuntimeError as error:
    print("raised as expected:", error)

[wreck] = store.find(healthy=False)
print("unhealthy node:", wreck.node_id, "| evidence:", (wreck / "partial.csv").read_text())

raised as expected: simulated mid-run failure
unhealthy node: 8339ddec | evidence: half-written


## Prune and compact
DAG-aware deletion (preview by default), then one verb to reclaim space.

In [7]:
preview = store.prune(wreck)  # dry run by default
print("would delete:", [n.node_id for n in preview])
deleted = store.prune(wreck, dry_run=False)
print("deleted:", [n.node_id for n in deleted])
print("chunks reclaimed by compact():", store.compact())

would delete: ['8339ddec']
deleted: ['8339ddec']
chunks reclaimed by compact(): 1


## Read-only SQL — the escape hatch
The schema is a public contract; the connection is `query_only`, so this can never write.

In [8]:
for row in store.sql(
    "SELECT step_type, count(*) AS n, sum(size_bytes) AS bytes "
    "FROM node GROUP BY step_type ORDER BY n DESC"
):
    print(f"{row['step_type']:<8} n={row['n']}  bytes={row['bytes']:,}")

ingest   n=5  bytes=600,024
clean    n=1  bytes=18
model    n=1  bytes=50,000


## The two outputs
A view-only HTML snapshot (shareable), and grep-able JSON sidecars (file portability, AD9).

In [9]:
snapshot = store.generate_web_graph()
sidecars = store.export()
print("snapshot:", snapshot.name, f"({snapshot.stat().st_size // 1024} KiB)")
example = next(sidecars.rglob("meta.json"))
print("sidecar sample keys:", sorted(json.loads(example.read_text()))[:6], "…")

store.close()
shutil.rmtree(workdir)
print("cleaned up.")

snapshot: interactive_pipeline.html (474 KiB)
sidecar sample keys: ['artifacts', 'content_hash', 'created_utc', 'duration_s', 'generation', 'healthy'] …
cleaned up.


---
The searchable explorer (search, node diff, runs table) is the **live server** — `store.host_live_graph()` / `python -m ancestree serve` — arriving in Phase 8; the snapshot above stays deliberately view-only so the query grammar exists exactly once.